In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [4]:
!pip install mapply

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 97.8 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompat

In [5]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import torch
import mapply
mapply.init(n_workers=-1,progressbar=True)
tqdm.pandas()

class Solver:
    
    def __init__(self,model="facebook/bart-large-mnli"):
        self.model = model
        
    def solve(self, text, labels):
        classifier = pipeline(
            "zero-shot-classification",
            model = self.model
        )
        result = classifier(text, labels)
        return result

In [6]:
device_id = 0 if torch.cuda.is_available() else -1

In [7]:
device_id

0

In [8]:
def get_top3(prompt, options, solver):

    result = solver.solve(prompt, options)
    winning_option_texts = result['labels']
    keys = ['A', 'B', 'C', 'D', 'E']
    top3_keys = []
    for text in winning_option_texts[:3]:
        original_index = options.index(text)
        top3_keys.append(keys[original_index])
        
    return " ".join(top3_keys)


solver = Solver(model='MoritzLaurer/deberta-v3-large-zeroshot-v2.0')
test_df['Prediction'] = test_df.progress_apply(lambda x: get_top3(x['prompt'],[x['A'],x['B'],x['C'],x['D'],x['E']],solver), axis = 1)

  0%|          | 0/500 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

  1%|▏         | 7/500 [00:24<21:54,  2.67s/it]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

  1%|▏         | 7/500 [00:26<31:24,  3.82s/it]


KeyboardInterrupt: 

In [ ]:
sub_df = test_df[['id','Prediction']]
sub_df.columns = ['ID','Prediction']
sub_df.to_csv('submission.csv',index=False)

In [ ]:
sub_df

In [ ]:
sub_df.Prediction.unique()